## Paso 0 — Sello de trazabilidad de los 17 archivos crudos

Este EDA usa 17 archivos del snapshot ya congelado `01-set-final/v1.0.0-piloto/`
(13 del lado propio de CabinaFletes + 4 del lado externo SICE-TAC/SAP).
Calculamos el hash MD5 de cada uno — una huella digital del archivo completo.
Como es la primera corrida, todavía no hay un valor "esperado" contra el cual
comparar; lo fijamos en el próximo paso, una vez veamos el resultado.


In [1]:
import hashlib
from datetime import datetime
from pathlib import Path

import pandas as pd

RUTA_DATOS_ORIGEN = Path("../../01-set-final/v1.0.0-piloto/datos")

ARCHIVOS_BD_PROPIA = [
    "bd-propia/audit_log.csv",
    "bd-propia/catalogo_clientes.csv",
    "bd-propia/catalogo_configuraciones_vehiculo.csv",
    "bd-propia/catalogo_rutas.csv",
    "bd-propia/catalogo_servicios_conexos.csv",
    "bd-propia/catalogo_usuarios.csv",
    "bd-propia/cotizacion_conexos.csv",
    "bd-propia/cotizacion_lineas.csv",
    "bd-propia/cotizaciones.csv",
    "bd-propia/inconsistencias.csv",
    "bd-propia/lineas_solicitud.csv",
    "bd-propia/solicitudes.csv",
    "bd-propia/tarifas_historicas.csv",
]

ARCHIVOS_FUENTES_CORPORATIVAS = [
    "fuentes-corporativas/catalogo-clientes-BUT000-organizaciones.csv",
    "fuentes-corporativas/historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy.csv",
    "fuentes-corporativas/piso-sicetac-tblRutasSicetac-muestra-202608.csv",
    "fuentes-corporativas/tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.csv",
]

ARCHIVOS_CRUDOS = ARCHIVOS_BD_PROPIA + ARCHIVOS_FUENTES_CORPORATIVAS
assert len(ARCHIVOS_CRUDOS) == 17, f"Se esperaban 17 archivos, hay {len(ARCHIVOS_CRUDOS)}"

filas_manifiesto = []
for nombre_relativo in ARCHIVOS_CRUDOS:
    ruta = RUTA_DATOS_ORIGEN / nombre_relativo
    contenido = ruta.read_bytes()
    filas_manifiesto.append({
        "archivo": nombre_relativo,
        "peso_kb": round(len(contenido) / 1024, 1),
        "hash_md5": hashlib.md5(contenido).hexdigest(),
    })

df_manifiesto = pd.DataFrame(filas_manifiesto)

print(f"Ejecutado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Archivos verificados: {len(df_manifiesto)}")
df_manifiesto


Ejecutado: 2026-09-11 14:15:01
Archivos verificados: 17


,archivo,peso_kb,hash_md5
0,bd-propia/audit_log.csv,11.1,84a042ca6bc389e303686b35d707826c
1,bd-propia/catalogo_clientes.csv,3.4,0775cfe522b35ce8a673b69582961bf1
2,bd-propia/catalogo_configuraciones_vehiculo.csv,2.1,01a5e6388da1331c24a57c2bb5eb55b1
3,bd-propia/catalogo_rutas.csv,4.1,ffff90b00d91fa2d8813526e1b7b8d86
4,bd-propia/catalogo_servicios_conexos.csv,1.8,862bc199f04d1bc78b3a4ca2b5d91ef1
5,bd-propia/catalogo_usuarios.csv,1.0,18847611ae531e13cb4283691fc1ad52
6,bd-propia/cotizacion_conexos.csv,1.0,8cca2ede059d72b4b556c9ea0f9e1a9d
7,bd-propia/cotizacion_lineas.csv,8.8,fdc70e7fb689283d5b8aded7a7796a07
8,bd-propia/cotizaciones.csv,3.1,b0ef7bdfb0d3ce7737d2413c4177441a
9,bd-propia/inconsistencias.csv,0.1,ce9452789a95fca9f466bab07ed3ff5a


In [2]:
HASHES_ESPERADOS = {
    # calculados el 2026-09-10, primera corrida de este notebook
    "bd-propia/audit_log.csv": "84a042ca6bc389e303686b35d707826c",
    "bd-propia/catalogo_clientes.csv": "0775cfe522b35ce8a673b69582961bf1",
    "bd-propia/catalogo_configuraciones_vehiculo.csv": "01a5e6388da1331c24a57c2bb5eb55b1",
    "bd-propia/catalogo_rutas.csv": "ffff90b00d91fa2d8813526e1b7b8d86",
    "bd-propia/catalogo_servicios_conexos.csv": "862bc199f04d1bc78b3a4ca2b5d91ef1",
    "bd-propia/catalogo_usuarios.csv": "18847611ae531e13cb4283691fc1ad52",
    "bd-propia/cotizacion_conexos.csv": "8cca2ede059d72b4b556c9ea0f9e1a9d",
    "bd-propia/cotizacion_lineas.csv": "fdc70e7fb689283d5b8aded7a7796a07",
    "bd-propia/cotizaciones.csv": "b0ef7bdfb0d3ce7737d2413c4177441a",
    "bd-propia/inconsistencias.csv": "ce9452789a95fca9f466bab07ed3ff5a",
    "bd-propia/lineas_solicitud.csv": "4f1aa48175509c631b262a16294c6342",
    "bd-propia/solicitudes.csv": "64752186a5269284b9ff8ddf332ca899",
    "bd-propia/tarifas_historicas.csv": "c2a75cb355a43cf2c4988c76994e4e3a",
    "fuentes-corporativas/catalogo-clientes-BUT000-organizaciones.csv": "501ecce38131e2f75e8f4baf90f853c9",
    "fuentes-corporativas/historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy.csv": "079af8714a622d2b843aea137fbfb0c4",
    "fuentes-corporativas/piso-sicetac-tblRutasSicetac-muestra-202608.csv": "d836788f2d8200de2e65853771c52ab4",
    "fuentes-corporativas/tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.csv": "bfbda2a996b681b4001fbc0b3c17cd31",
}

df_manifiesto["hash_esperado"] = df_manifiesto["archivo"].map(HASHES_ESPERADOS)
df_manifiesto["coincide"] = df_manifiesto["hash_md5"] == df_manifiesto["hash_esperado"]

archivos_cambiados = df_manifiesto.loc[~df_manifiesto["coincide"], "archivo"].tolist()
assert not archivos_cambiados, f"¡Estos archivos cambiaron desde la última corrida! {archivos_cambiados}"

print("Los 17 archivos coinciden con el hash esperado — nada cambió desde la última corrida.")


Los 17 archivos coinciden con el hash esperado — nada cambió desde la última corrida.


## Paso 1 — Fila fantasma en los 17 archivos

El plan detectó, durante el encuadre, que al menos 3 archivos traían una fila
extra de valores tipo `"------"` — un artefacto de exportación (parece una
fila separadora de tabla Markdown que se coló como si fuera un dato real).
Aquí lo verificamos en **los 17 archivos**, no solo en los 3 ya vistos:
comparamos cuántas filas dice el acta de entrega que debería tener cada
archivo contra cuántas filas realmente lee pandas, y revisamos si la primera
fila de datos (justo después del encabezado) es esa fila fantasma. Donde
aparezca, la excluimos explícitamente antes de seguir — no se analiza como
si fuera un registro real.


In [3]:
CONTEO_DOCUMENTADO = {
    "bd-propia/solicitudes.csv": 13,
    "bd-propia/lineas_solicitud.csv": 43,
    "bd-propia/cotizaciones.csv": 18,
    "bd-propia/cotizacion_lineas.csv": 48,
    "bd-propia/cotizacion_conexos.csv": 7,
    "bd-propia/tarifas_historicas.csv": 15,
    "bd-propia/audit_log.csv": 72,
    "bd-propia/inconsistencias.csv": 1,
    "bd-propia/catalogo_clientes.csv": 21,
    "bd-propia/catalogo_rutas.csv": 31,
    "bd-propia/catalogo_configuraciones_vehiculo.csv": 16,
    "bd-propia/catalogo_servicios_conexos.csv": 11,
    "bd-propia/catalogo_usuarios.csv": 11,
    "fuentes-corporativas/piso-sicetac-tblRutasSicetac-muestra-202608.csv": 151,
    "fuentes-corporativas/tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.csv": 23_020,
    "fuentes-corporativas/historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy.csv": 25_318,
    "fuentes-corporativas/catalogo-clientes-BUT000-organizaciones.csv": 38_995,
}

import re

PATRON_SEPARADOR = re.compile(r"^-{2,}$")

def es_fila_separador(fila):
    valores = [str(v).strip() for v in fila if str(v).strip() != ""]
    if not valores:
        return False
    return all(PATRON_SEPARADOR.match(v) for v in valores)

filas_revision = []
tablas_crudas = {}

for nombre_relativo in ARCHIVOS_CRUDOS:
    ruta = RUTA_DATOS_ORIGEN / nombre_relativo
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8")

    primera_fila = df.iloc[0] if len(df) > 0 else None
    tiene_fantasma = es_fila_separador(primera_fila) if primera_fila is not None else False

    if tiene_fantasma:
        df = df.iloc[1:].reset_index(drop=True)

    tablas_crudas[nombre_relativo] = df

    documentado = CONTEO_DOCUMENTADO.get(nombre_relativo)
    filas_revision.append({
        "archivo": nombre_relativo,
        "filas_documentadas": documentado,
        "fila_fantasma_detectada": tiene_fantasma,
        "filas_tras_excluir_fantasma": len(df),
        "coincide_con_documentado": (documentado is None) or (len(df) == documentado),
    })

df_revision = pd.DataFrame(filas_revision)
df_revision


,archivo,filas_documentadas,fila_fantasma_detectada,filas_tras_excluir_fantasma,coincide_con_documentado
0,bd-propia/audit_log.csv,72,True,72,True
1,bd-propia/catalogo_clientes.csv,21,True,21,True
2,bd-propia/catalogo_configuraciones_vehiculo.csv,16,True,16,True
3,bd-propia/catalogo_rutas.csv,31,True,31,True
4,bd-propia/catalogo_servicios_conexos.csv,11,True,11,True
5,bd-propia/catalogo_usuarios.csv,11,True,11,True
6,bd-propia/cotizacion_conexos.csv,7,True,7,True
7,bd-propia/cotizacion_lineas.csv,48,True,48,True
8,bd-propia/cotizaciones.csv,18,True,18,True
9,bd-propia/inconsistencias.csv,1,True,1,True


## Paso 2 — Separar el "núcleo" de los casos "raros" en piso-sicetac

`piso-sicetac-tblRutasSicetac-muestra-202608.csv` no es una muestra al azar
de SICE-TAC: alguien la armó a propósito con 126 filas típicas (`nucleo`) +
25 filas elegidas deliberadamente por ser casos extremos (`rara_*`: valor
máximo, valor mínimo, ambigüedad de ruta, horas máximas, vía no estándar).
Si calculamos cualquier promedio o distribución sobre el archivo completo sin
separar estos dos grupos, el resultado sale sesgado hacia los extremos.
Separamos esto **desde la carga** (no más adelante) para que ningún paso
futuro pueda mezclarlos sin querer.


In [4]:
CLAVE_SICETAC = "fuentes-corporativas/piso-sicetac-tblRutasSicetac-muestra-202608.csv"

df_sicetac = tablas_crudas[CLAVE_SICETAC]
print(df_sicetac["Muestra"].value_counts())

sicetac_nucleo = df_sicetac[df_sicetac["Muestra"] == "nucleo"].reset_index(drop=True)
sicetac_raras = df_sicetac[df_sicetac["Muestra"] != "nucleo"].reset_index(drop=True)

assert len(sicetac_nucleo) == 126, f"nucleo esperado 126, salió {len(sicetac_nucleo)}"
assert len(sicetac_raras) == 25, f"raras esperado 25, salió {len(sicetac_raras)}"

# Quitamos la versión combinada del diccionario a propósito — de aquí en
# adelante solo deben usarse las dos partes por separado, nunca mezcladas.
del tablas_crudas[CLAVE_SICETAC]
tablas_crudas["fuentes-corporativas/piso-sicetac__nucleo"] = sicetac_nucleo
tablas_crudas["fuentes-corporativas/piso-sicetac__raras"] = sicetac_raras

print(f"nucleo: {len(sicetac_nucleo)} filas — usar para cualquier estadístico 'típico'")
print(f"raras:  {len(sicetac_raras)} filas — casos extremos, se analizan aparte")


Muestra
nucleo                                       126
rara_ambiguedad_567_candidatos_misma_ruta      5
rara_valor_maximo                              5
rara_valor_minimo                              5
rara_via_no_estandar                           5
rara_horas_recorrido_maximas                   5
Name: count, dtype: int64
nucleo: 126 filas — usar para cualquier estadístico 'típico'
raras:  25 filas — casos extremos, se analizan aparte


## Paso 3 — Tipamos cada columna a su tipo real (18 tablas)

Hasta ahora todo sigue siendo texto plano (`dtype=str`), a propósito, para
poder revisar la fila fantasma sin que pandas adivinara nada. Ahora sí
convertimos cada columna a lo que realmente es (fecha, número, sí/no, o
texto/código que se queda igual) — usando lo documentado en
`diccionario-y-linaje.md`, y donde no alcanzaba (los 5 catálogos), lo que
confirmamos revisando el CSV real. De paso resolvemos que CabinaFletes
marca lo vacío como el texto `"NULL"` y SAP/SICE-TAC lo dejan en blanco —
ambos se convierten al mismo valor de "nulo" real de pandas.


In [5]:
TIPOS_POR_TABLA = {
    "bd-propia/audit_log.csv": {
        "Id": "guid", "UsuarioId": "guid", "Entidad": "categorico", "EntidadId": "guid",
        "Accion": "categorico", "CreatedAt": "fecha",
    },
    "bd-propia/catalogo_clientes.csv": {
        "Id": "guid", "Nombre": "texto", "Nit": "texto", "RazonSocial": "texto",
        "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/catalogo_configuraciones_vehiculo.csv": {
        "Id": "guid", "TipoVehiculo": "categorico", "Carroceria": "categorico",
        "RangoPeso": "texto", "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/catalogo_rutas.csv": {
        "Id": "guid", "Origen": "texto", "Destino": "texto",
        "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/catalogo_servicios_conexos.csv": {
        "Id": "guid", "Nombre": "texto", "Categoria": "categorico", "ValorBase": "numero",
        "VigenteHasta": "fecha", "Activo": "booleano_01", "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/catalogo_usuarios.csv": {
        "Id": "guid", "Rol": "categorico", "Activo": "booleano_01", "CreatedAt": "fecha",
    },
    "bd-propia/cotizacion_conexos.csv": {
        "Id": "guid", "CotizacionId": "guid", "ServicioConexoId": "guid",
        "Cantidad": "numero", "ValorAplicado": "numero",
    },
    "bd-propia/cotizacion_lineas.csv": {
        "Id": "guid", "CotizacionId": "guid", "LineaSolicitudId": "guid", "ValorFlete": "numero",
        "ValorAprovisionamiento": "numero", "ValorRetornoContenedor": "numero", "Fuente": "categorico",
        "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/cotizaciones.csv": {
        "Id": "guid", "SolicitudId": "guid", "Estado": "categorico", "Resultado": "categorico",
        "Total": "numero", "EntregadaPor": "guid", "EntregadaAt": "fecha",
        "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/inconsistencias.csv": {
        "Entidad": "categorico", "EntidadId": "guid", "Motivo": "texto",
    },
    "bd-propia/lineas_solicitud.csv": {
        "Id": "guid", "SolicitudId": "guid", "RutaId": "guid", "ConfigVehiculoId": "guid",
        "Tipologia": "categorico", "LineaNegocio": "categorico", "ClaseViaje": "categorico",
        "ClaseContenedor": "categorico", "Exclusivo": "booleano_01", "Consolidado": "booleano_01",
        "RetornaContenedor": "booleano_01", "LevanteContenedorDiferente": "booleano_01",
        "ValorAprovisionamiento": "numero", "Peso": "numero", "ValorMercancia": "numero",
        "EstadoLinea": "categorico", "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/solicitudes.csv": {
        "Id": "guid", "ClienteId": "guid", "Tipo": "categorico", "Modalidad": "categorico",
        "Deadline": "fecha", "ProbabilidadCierre": "numero", "ValorEstimado": "numero",
        "ScorePrioridad": "numero", "Estado": "categorico", "CreadoPor": "guid",
        "CreatedAt": "fecha", "UpdatedAt": "fecha",
    },
    "bd-propia/tarifas_historicas.csv": {
        "Id": "guid", "RutaId": "guid", "ConfigVehiculoId": "guid", "Tipologia": "categorico",
        "Valor": "numero", "Fecha": "fecha", "CotizacionId": "guid",
    },
    "fuentes-corporativas/catalogo-clientes-BUT000-organizaciones.csv": {
        "PARTNER": "guid", "TYPE": "categorico", "BU_GROUP": "categorico",
        "NAME_ORG1": "texto", "NAME_ORG2": "texto", "TAXNUM": "texto", "TAXNUMXL": "texto",
        "TEL_NUMBER": "texto", "CRDAT": "fecha_dayfirst", "VALID_TO": "fecha_dayfirst",
    },
    "fuentes-corporativas/historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy.csv": {
        "ID": "guid", "ID_TARIFA": "guid", "LINEANEGOCIO": "categorico",
        "LineaNegocioDescripcion": "texto", "ORIGEN": "categorico", "DESTINO": "categorico",
        "VALOR": "numero", "VIGENCIA": "fecha", "CLASECONTENEDOR": "categorico",
        "TAMANOCONTENEDOR": "categorico", "RANGOPESOINF": "numero", "RANGOPESOSUP": "numero",
        "TIPOVEHICULO": "categorico", "VehiculoDescripcion": "texto", "TIPOCARROSERIA": "categorico",
        "PESOLIMITE": "numero", "FECHACREACION": "fecha", "USUARIOCREACION": "texto",
        "CLASEVIAJE": "categorico", "DEVOLUCION": "booleano_textotf",
    },
    "fuentes-corporativas/tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy.csv": {
        "ID": "guid", "LINEANEGOCIO": "categorico", "LineaNegocioDescripcion": "texto",
        "ORIGEN": "categorico", "DESTINO": "categorico", "VALOR": "numero", "VIGENCIA": "fecha",
        "CLASECONTENEDOR": "categorico", "TAMANOCONTENEDOR": "categorico",
        "RANGOPESOINF": "numero", "RANGOPESOSUP": "numero", "TIPOVEHICULO": "categorico",
        "VehiculoDescripcion": "texto", "TIPOCARROSERIA": "categorico", "PESOLIMITE": "numero",
        "FECHACREACION": "fecha", "USUARIOCREACION": "texto", "CLASEVIAJE": "categorico",
        "DEVOLUCION": "booleano_textotf", "FECHAMODIFICACION": "fecha",
        "USUARIOMODIFICACION": "texto", "DETALLESTARIFA": "texto",
        "CONSOLIDADO": "booleano_textotf", "EXCLUSIVO": "booleano_textotf",
    },
}

ESQUEMA_SICETAC = {
    "ID": "guid", "PERIODO": "categorico", "CONFIGURACION": "categorico", "VALOR": "numero",
    "NOMBRERUTA": "texto", "NOMBRETIPOCARGA": "categorico", "NOMBREUNIDADTRANSPORTE": "categorico",
    "ORIGEN": "categorico", "DESTINO": "categorico", "VALORHORA": "numero",
    "VALORSICECONHORA": "numero", "HORASRECORRIDO": "numero", "HORASPACTADAS": "numero",
    "CONDICIONCARGA": "categorico", "VIAESTANDAR": "booleano_sino", "Muestra": "categorico",
}
TIPOS_POR_TABLA["fuentes-corporativas/piso-sicetac__nucleo"] = ESQUEMA_SICETAC
TIPOS_POR_TABLA["fuentes-corporativas/piso-sicetac__raras"] = ESQUEMA_SICETAC


def convertir_columna(serie, tipo):
    if tipo == "fecha":
        return pd.to_datetime(serie, errors="coerce")
    if tipo == "fecha_dayfirst":
        return pd.to_datetime(serie, errors="coerce", dayfirst=True)
    if tipo == "numero":
        return pd.to_numeric(serie, errors="coerce").astype(float)
    if tipo == "booleano_01":
        return serie.str.strip().map({"0": False, "1": True})
    if tipo == "booleano_sino":
        return serie.str.strip().str.upper().map({"SI": True, "NO": False})
    if tipo == "booleano_textotf":
        return serie.str.strip().map({"True": True, "False": False})
    # guid, texto, categorico: se quedan como texto; solo limpiamos espacios y "nulos" textuales
    limpio = serie.str.strip()
    return limpio.replace({"NULL": pd.NA, "": pd.NA})


resumen_tipado = []
for clave, df in tablas_crudas.items():
    esquema = TIPOS_POR_TABLA[clave]
    faltantes = set(df.columns) - set(esquema.keys())
    sobrantes = set(esquema.keys()) - set(df.columns)
    assert not faltantes, f"{clave}: columnas sin tipo asignado: {faltantes}"
    assert not sobrantes, f"{clave}: el esquema tiene columnas que no existen en el archivo: {sobrantes}"

    df_tipado = df.copy()
    for columna, tipo in esquema.items():
        df_tipado[columna] = convertir_columna(df[columna], tipo)
    tablas_crudas[clave] = df_tipado

    for columna, tipo in esquema.items():
        resumen_tipado.append({
            "tabla": clave,
            "columna": columna,
            "tipo_asignado": tipo,
            "dtype_final": str(df_tipado[columna].dtype),
            "nulos": int(df_tipado[columna].isna().sum()),
        })

df_resumen_tipado = pd.DataFrame(resumen_tipado)
print(f"Tablas tipadas: {len(tablas_crudas)}")
print(f"Columnas procesadas: {len(df_resumen_tipado)}")
df_resumen_tipado


C:\Users\Scalderon\AppData\Local\Temp\ipykernel_1548\2153181384.py:100: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(serie, errors="coerce")
C:\Users\Scalderon\AppData\Local\Temp\ipykernel_1548\2153181384.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(serie, errors="coerce", dayfirst=True)


Tablas tipadas: 18
Columnas procesadas: 184


,tabla,columna,tipo_asignado,dtype_final,nulos
0,bd-propia/audit_log.csv,Id,guid,str,0
1,bd-propia/audit_log.csv,UsuarioId,guid,str,0
2,bd-propia/audit_log.csv,Entidad,categorico,str,0
3,bd-propia/audit_log.csv,EntidadId,guid,str,0
4,bd-propia/audit_log.csv,Accion,categorico,str,0
...,...,...,...,...,...
179,fuentes-corporativas/piso-sicetac__raras,HORASRECORRIDO,numero,float64,0
180,fuentes-corporativas/piso-sicetac__raras,HORASPACTADAS,numero,float64,0
181,fuentes-corporativas/piso-sicetac__raras,CONDICIONCARGA,categorico,str,0
182,fuentes-corporativas/piso-sicetac__raras,VIAESTANDAR,booleano_sino,bool,0


In [6]:
df_resumen_tipado[df_resumen_tipado["nulos"] > 0].sort_values(["tabla", "columna"])


,tabla,columna,tipo_asignado,dtype_final,nulos
9,bd-propia/catalogo_clientes.csv,RazonSocial,texto,str,2
14,bd-propia/catalogo_configuraciones_vehiculo.csv,Carroceria,categorico,str,1
15,bd-propia/catalogo_configuraciones_vehiculo.csv,RangoPeso,texto,str,7
27,bd-propia/catalogo_servicios_conexos.csv,VigenteHasta,fecha,datetime64[us],1
44,bd-propia/cotizacion_lineas.csv,ValorAprovisionamiento,numero,float64,48
45,bd-propia/cotizacion_lineas.csv,ValorRetornoContenedor,numero,float64,48
55,bd-propia/cotizaciones.csv,EntregadaAt,fecha,datetime64[us],12
54,bd-propia/cotizaciones.csv,EntregadaPor,guid,str,12
68,bd-propia/lineas_solicitud.csv,ClaseContenedor,categorico,str,41
67,bd-propia/lineas_solicitud.csv,ClaseViaje,categorico,str,38


## Paso 4 — Normalizamos nombres de columna a camelCase (18 tablas)

Igual criterio que el EDA-1: encabezados a camelCase (primera palabra en
minúscula, siguientes con inicial en mayúscula, sin tildes ni espacios),
para que el código de los próximos notebooks sea menos frágil y comparable
entre ambos EDA. Ya con las 18 tablas tipadas (Paso 3), repetimos el mismo
criterio en cada una y guardamos el mapeo original → nuevo para trazabilidad.

In [7]:
import re
import unicodedata

def a_camel_case(nombre):
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii")
    con_separadores = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", sin_tildes)
    palabras = [p for p in re.split(r"[^a-zA-Z0-9]+", con_separadores) if p]
    primera, *resto = palabras
    return primera.lower() + "".join(p.capitalize() for p in resto)

mapeos_por_tabla = []

for clave, df in tablas_crudas.items():
    mapeo = {col: a_camel_case(col) for col in df.columns}
    nombres_nuevos = list(mapeo.values())
    assert len(nombres_nuevos) == len(set(nombres_nuevos)), f"Colisión de nombres en {clave}: {mapeo}"
    tablas_crudas[clave] = df.rename(columns=mapeo)
    for original, nuevo in mapeo.items():
        mapeos_por_tabla.append({"tabla": clave, "columna_original": original, "columna_nueva": nuevo})

df_mapeo_columnas = pd.DataFrame(mapeos_por_tabla)
print(f"Tablas procesadas: {len(tablas_crudas)}")
print(f"Columnas renombradas en total: {len(df_mapeo_columnas)}")
df_mapeo_columnas

Tablas procesadas: 18
Columnas renombradas en total: 184


,tabla,columna_original,columna_nueva
0,bd-propia/audit_log.csv,Id,id
1,bd-propia/audit_log.csv,UsuarioId,usuarioId
2,bd-propia/audit_log.csv,Entidad,entidad
3,bd-propia/audit_log.csv,EntidadId,entidadId
4,bd-propia/audit_log.csv,Accion,accion
...,...,...,...
179,fuentes-corporativas/piso-sicetac__raras,HORASRECORRIDO,horasrecorrido
180,fuentes-corporativas/piso-sicetac__raras,HORASPACTADAS,horaspactadas
181,fuentes-corporativas/piso-sicetac__raras,CONDICIONCARGA,condicioncarga
182,fuentes-corporativas/piso-sicetac__raras,VIAESTANDAR,viaestandar


## Paso 5 — Guardamos las 18 tablas ya limpias en Parquet

Cada tabla (sin fila fantasma, con piso-sicetac separado en núcleo/raras, y
columnas en camelCase) se guarda en `datos/interim/`, un archivo Parquet por
tabla — sin aplanar, como pide el plan (§2.1). Los notebooks siguientes
parten de aquí, no de los CSV originales.


In [8]:
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
RUTA_INTERIM.mkdir(parents=True, exist_ok=True)

def nombre_parquet(clave):
    base = clave.split("/")[-1]
    base = base.replace(".csv", "")
    base = base.replace("__", "_")
    return base + ".parquet"

resumen_guardado = []
for clave, df in tablas_crudas.items():
    nombre = nombre_parquet(clave)
    ruta_salida = RUTA_INTERIM / nombre
    df.to_parquet(ruta_salida, index=False)
    resumen_guardado.append({
        "tabla_origen": clave,
        "archivo_parquet": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
    })

df_resumen_guardado = pd.DataFrame(resumen_guardado)
print(f"{len(df_resumen_guardado)} archivos Parquet guardados en {RUTA_INTERIM}/")
df_resumen_guardado


18 archivos Parquet guardados en ..\datos\interim/


,tabla_origen,archivo_parquet,filas,columnas
0,bd-propia/audit_log.csv,audit_log.parquet,72,6
1,bd-propia/catalogo_clientes.csv,catalogo_clientes.parquet,21,6
2,bd-propia/catalogo_configuraciones_vehiculo.csv,catalogo_configuraciones_vehiculo.parquet,16,6
3,bd-propia/catalogo_rutas.csv,catalogo_rutas.parquet,31,5
4,bd-propia/catalogo_servicios_conexos.csv,catalogo_servicios_conexos.parquet,11,8
5,bd-propia/catalogo_usuarios.csv,catalogo_usuarios.parquet,11,4
6,bd-propia/cotizacion_conexos.csv,cotizacion_conexos.parquet,7,5
7,bd-propia/cotizacion_lineas.csv,cotizacion_lineas.parquet,48,9
8,bd-propia/cotizaciones.csv,cotizaciones.parquet,18,9
9,bd-propia/inconsistencias.csv,inconsistencias.parquet,1,3


## Síntesis del notebook 00 — Montaje reproducible

**Objetivo cumplido (Fase 1, §2 del plan):** dejar las 17 tablas de origen
cargadas, verificadas, tipadas y guardadas de forma reproducible, sin
depender de rutas locales ni de confiar a ciegas en lo que pandas infiera
al leer los CSV.

**Trazabilidad (Paso 0):** hash MD5 de los 17 archivos crudos registrado y
verificado. Si alguien reemplaza uno solo de ellos, este notebook va a
fallar de forma explícita en la primera celda, no en silencio más adelante.

**Fila fantasma (Paso 1):** el plan había detectado el problema en 3
archivos durante el encuadre; verificado en los 17, apareció en **14 de
17** — más de lo esperado. Corrección al plan: no está al final del
archivo como se asumía, sino **justo después del encabezado** (parece una
fila separadora de tabla Markdown que se coló como dato real). Excluida
explícitamente en los 14, con evidencia (conteo de filas vs. acta de
entrega). Candidato a hallazgo H-001 en `reportes/hallazgos.md`.

**Separación núcleo/raras de piso-sicetac (Paso 2):** confirmado 126 filas
`nucleo` + 25 `rara_*` (5 categorías de 5), tal como documentaba el plan
(§1.5). Guardadas como dos tablas independientes desde este punto — la
versión combinada se descarta a propósito para que ningún paso futuro las
mezcle sin querer.

**Tipado (Paso 3):** 184 columnas tipadas en las 18 tablas, usando
`diccionario-y-linaje.md` como fuente — y donde no alcanzaba (los 5
catálogos, cuya documentación resultó estar incompleta incluso en el
archivo de origen), inspección directa del CSV real. Dos hallazgos nuevos
en el camino:
- `CabinaFletes` marca lo vacío con el texto literal `"NULL"`; SAP/SICE-TAC
  lo dejan en blanco — ambas convenciones normalizadas al mismo valor nulo
  real de pandas.
- `CRDAT`/`VALID_TO` de `catalogo-clientes-BUT000-organizaciones.csv`
  vienen en formato colombiano día/mes (`DD/MM/AAAA`), a diferencia de todo
  el resto del set que usa ISO (`AAAA-MM-DD`) — sin `dayfirst=True`,
  pandas asume mes/día por defecto y invertiría fechas ambiguas en
  silencio. Corregido antes de guardar.

Dos observaciones que quedan para `reportes/hallazgos.md`/`limitaciones.md`
más adelante, no resueltas aquí:
- `TAXNUMXL` (llave de cruce documentada junto con `TAXNUM`) sale **100%
  nula** en las 38.995 filas — no es utilizable en la práctica como llave.
- El archivo `DICCIONARIO_DATOS.md`, referenciado por
  `diccionario-y-linaje.md` para los 5 catálogos, **no existe** dentro del
  snapshot congelado `v1.0.0-piloto/` — rompe la promesa de que un snapshot
  entregado sea autocontenido (sí existe, sin cambios, en `00-fuentes/`).

**Normalización de nombres (Paso 4):** columnas convertidas a camelCase en
las 18 tablas, mismo criterio que EDA-1, sin colisiones de nombres en
ninguna.

**Persistencia (Paso 5):** las 18 tablas guardadas en
`datos/interim/*.parquet`, verificado que cada una conserva exactamente el
número de filas documentado en el acta de entrega (o, para `piso-sicetac`,
el conteo esperado de núcleo/raras).

**Qué le queda listo al siguiente notebook:** 18 Parquet confiables,
tipados, con nombres de columna estables y sin la fila fantasma —
`01_perfilado_estructural.ipynb` (Fase 2 del plan) parte de ahí
directamente, sin volver a tocar los CSV originales.
